In [ ]:
!pip install httpx beautifulsoup4 pandas lxml tqdm selenium webdriver_manager

In [ ]:
import os

# Download the Google Chrome signing key
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | sudo apt-key add -

# Add the Google Chrome repository to apt sources
!echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" | sudo tee /etc/apt/sources.list.d/google-chrome.list

# Update apt packages and install google-chrome-stable
!sudo apt-get update -qq
!sudo apt-get install -y google-chrome-stable

OK
deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main
W: http://dl.google.com/linux/chrome/deb/dists/stable/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0 libatk1.0-0
  libatk1.0-data libatspi2.0-0 libvulkan1 libxcomposite1 libxtst6
  mesa-vulkan-drivers session-migration
The following NEW packages will be installed:
  at-spi2-core google-chrome-stable gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk1.0-0 libatk1.0-data libatspi2.0-0 libvulkan1
  libxcomposite1 libxtst6 mesa-vulkan-drive

In [ ]:
import time
import random
import json
import re
import os
import logging
from pathlib import Path

import requests
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [ ]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger(__name__)

BASE_URL       = "https://nationalgeographic.grid.id"
SEARCH_URL     = f"{BASE_URL}/search?keyword=sejarah"
OUTPUT_CSV     = "natgeo_sejarah.csv"
PROGRESS_FILE  = "natgeo_progress.json"
DELAY_MIN      = 1.5
DELAY_MAX      = 3.5
MAX_RETRIES    = 3
TIMEOUT        = 20
MAX_ARTICLES_TO_SCRAPE = 1000

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Referer": BASE_URL,
}

CONTENT_JUNK_SELECTORS = [
    "script", "style", "noscript", "iframe", "ins",
    "figcaption",
    "[class*='caption']",
    "[class*='ads']", "[class*='ad-']", "[id*='ads']",
    "[class*='promo']", "[class*='iklan']",
    "[class*='share']", "[class*='social']",
    "[class*='related']", "[class*='rekomendasi']",
    "[class*='baca-juga']",
    "[class*='widget']",
    "[class*='author']", "[class*='profile']",
    "[class*='tag']",
    "[class*='premium']", "[class*='subscribe']", "[class*='paywall']",
]

JUNK_TEXT_RE = re.compile(
    r"^(baca\s+juga|lihat\s+juga|baca\s+pula|artikel\s+terkait"
    r"|sumber\s*:|referensi\s*:|advertisement|iklan|foto\s*:|keterangan\s+foto"
    r"|nationalgeographic\\.co\\.id—|nationalgeographic\\.co\\.id\s*—)",
    re.IGNORECASE,
)

In [ ]:
from bs4 import Tag
from typing import Union
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from tqdm.auto import tqdm
import csv

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    return s

def safe_get(session: requests.Session, url: str) -> requests.Response | None:
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = session.get(url, timeout=TIMEOUT)
            if resp.status_code == 200:
                return resp
            elif resp.status_code == 429:
                wait = 30 * attempt
                log.warning(f"Rate-limited. Tunggu {wait}s …")
                time.sleep(wait)
            else:
                log.warning(f"HTTP {resp.status_code} — {url} (attempt {attempt})")
                time.sleep(5 * attempt)
        except Exception as e:
            log.error(f"Request error attempt {attempt}: {e}")
            time.sleep(5 * attempt)
    return None

def polite_delay():
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

def make_driver() -> webdriver.Chrome:
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--window-size=1280,900")
    opts.add_argument(f"user-agent={HEADERS['User-Agent']}")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)
    return driver

def collect_article_urls(driver: webdriver.Chrome) -> list[str]:
    log.info("Membuka halaman pencarian …")
    driver.get(SEARCH_URL)
    time.sleep(3)

    urls: set[str] = set()
    click_count = 0

    while True:
        soup = BeautifulSoup(driver.page_source, "lxml")
        for a in soup.find_all("a", href=re.compile(r"read/\d+")):
            href = a["href"].split("?")[0].split("#")[0]
            if href.startswith("/"):
                href = BASE_URL + href
            urls.add(href)

        if len(urls) >= MAX_ARTICLES_TO_SCRAPE:
             log.info(f"Sudah mencapai target minimal URL: {len(urls)}")
             break

        try:
            btn = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable(
                    (By.XPATH, "//*[contains(text(),'Lihat Lainnya') or contains(text(),'lihat lainnya')]")
                )
            )
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
            time.sleep(1)
            driver.execute_script("arguments[0].click();", btn)
            click_count += 1
            print(f"\r[Selenium] Klik 'Lihat Lainnya' #{click_count} | URL terkumpul: {len(urls)}", end="", flush=True)
            time.sleep(random.uniform(2.0, 3.5))
        except Exception:
            print("\nTidak ada lagi tombol 'Lihat Lainnya' atau error.")
            break

    result = sorted(urls)
    log.info(f"Total URL unik ditemukan: {len(result)}")
    return result

def clean_element(element: Union[BeautifulSoup, Tag], selectors: list[str]) -> None:
    for sel in selectors:
        for tag in element.select(sel):
            tag.decompose()

def find_content_container(soup: BeautifulSoup):
    for pattern in [r"read__content", r"article.?content", r"content.?article", r"post.?body", r"detail.?content"]:
        tag = soup.find("div", class_=re.compile(pattern, re.I))
        if tag and tag.get_text(strip=True):
            return tag
    for tag_name in ["article", "main"]:
        tag = soup.find(tag_name)
        if tag and tag.get_text(strip=True):
            for pattern_inner in [r"read__content", r"article.?content"]:
                inner_tag = tag.find("div", class_=re.compile(pattern_inner, re.I))
                if inner_tag and inner_tag.get_text(strip=True):
                    return inner_tag
            return tag
    return None

def extract_paragraphs(container) -> list[str]:
    paragraphs = []
    for p in container.find_all("p"):
        text = p.get_text(separator=" ", strip=True)
        # Remove newlines and excess whitespace to ensure one-row format
        text = re.sub(r"\s+", " ", text)
        if len(text) < 40 or JUNK_TEXT_RE.match(text):
            continue
        paragraphs.append(text)
    return paragraphs

def scrape_article(session: requests.Session, url: str) -> dict | None:
    page_all_url = url + "?page=all"
    resp = safe_get(session, page_all_url)
    if resp is None: return None
    soup = BeautifulSoup(resp.text, "lxml")
    title_tag = soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else (soup.find("title").get_text(strip=True) if soup.find("title") else url)
    container = find_content_container(soup)
    if not container: return None
    clean_element(container, CONTENT_JUNK_SELECTORS)
    paragraphs = extract_paragraphs(container)
    # Joining with a single space to prevent CSV layout breaks
    content = " ".join(paragraphs)
    return {"url": url, "title": title, "content": content}

def load_progress() -> dict:
    if Path(PROGRESS_FILE).exists():
        with open(PROGRESS_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"scraped_urls": [], "article_urls": []}

def save_progress(progress: dict) -> None:
    with open(PROGRESS_FILE, "w", encoding="utf-8") as f:
        json.dump(progress, f, ensure_ascii=False, indent=2)

def main():
    progress = load_progress()
    if not progress["article_urls"]:
        log.info("Mengumpulkan URL via Selenium …")
        driver = make_driver()
        try:
            article_urls = collect_article_urls(driver)
        finally:
            driver.quit()
        progress["article_urls"] = article_urls
        save_progress(progress)
    else:
        article_urls = progress["article_urls"]

    scraped_set = set(progress["scraped_urls"])
    remaining = [u for u in article_urls if u not in scraped_set]
    total_to_scrape = min(len(remaining), MAX_ARTICLES_TO_SCRAPE)
    remaining_for_this_run = remaining[:total_to_scrape]

    log.info(f"Memulai scraping {total_to_scrape} artikel...")

    results = []
    if Path(OUTPUT_CSV).exists():
        try:
            results = pd.read_csv(OUTPUT_CSV, encoding="utf-8-sig").to_dict("records")
        except:
            results = []

    session = make_session()
    pbar = tqdm(total=total_to_scrape, desc="Scraping Progress")

    for idx, url in enumerate(remaining_for_this_run, start=1):
        article = scrape_article(session, url)
        if article:
            results.append(article)
            progress["scraped_urls"].append(url)
            if idx % 10 == 0 or idx == total_to_scrape:
                # Use quoting=csv.QUOTE_ALL (1) to keep content in one row
                pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig", quoting=csv.QUOTE_ALL)
                save_progress(progress)

        pbar.update(1)
        polite_delay()

    pbar.close()
    log.info(f"✅ Selesai! Data disimpan di {OUTPUT_CSV}")

In [ ]:
def test_single():
    session = make_session()
    url = "https://nationalgeographic.grid.id/read/134379718/mengapa-sejarah-kekaisaran-ottoman-masih-relevan-hingga-saat-ini"
    result = scrape_article(session, url)
    if result:
        print("URL   :", result["url"])
        print("Judul :", result["title"])
        print(f"Konten ({len(result['content'])} chars):")
        print("-" * 60)
        print(result["content"][:2000])
    else:
        print("Gagal.")

In [ ]:
if __name__ == "__main__":
    test_single()

URL   : https://nationalgeographic.grid.id/read/134379718/mengapa-sejarah-kekaisaran-ottoman-masih-relevan-hingga-saat-ini
Judul : Mengapa Sejarah Kekaisaran Ottoman Masih Relevan Hingga Saat Ini?
Konten (7495 chars):
------------------------------------------------------------
Nationalgeographic.co.id— Salah satu kekaisaran terbesar dan terlama dalam sejarah dunia, Kekaisaran Ottoman tetap signifikan secara historis hingga kini. Kebangkitannya menjadi kekuatan membantu membentuk era sejarah utama. Khususnya di Timur Tengah, Eropa tenggara, dan Mediterania timur.

Lebih jauh lagi, runtuhnya Kekaisaran Ottoman memengaruhi banyak perbatasan nasional modern di wilayah-wilayah ini. Terakhir, dampak Kekaisaran Ottoman terhadap identitas agama dan nasional terus dirasakan dalam politik dan budaya kontemporer.

Mengapa sejarah Kekaisaran Ottoman masih relevan hingga kini?

Penaklukan Konstantinopel oleh Ottoman pada tahun 1453 dianggap sebagai titik balik utama antara era abad pertengahan dan

In [ ]:
if __name__ == "__main__":
    main()

[Selenium] Klik 'Lihat Lainnya' #183 | URL terkumpul: 740
Tidak ada lagi tombol 'Lihat Lainnya' atau error.


Scraping Progress:   0%|          | 0/744 [00:00<?, ?it/s]

In [ ]:
import pandas as pd
import os

if os.path.exists(OUTPUT_CSV):
    df_scraped = pd.read_csv(OUTPUT_CSV)

    print(f"Total artikel yang berhasil di-scrape: {len(df_scraped)}")

    pd.set_option('display.max_colwidth', 100)

    display(df_scraped.head(5))
else:
    print(f"File {OUTPUT_CSV} belum ditemukan. Silakan jalankan cell main() terlebih dahulu.")

Total artikel yang berhasil di-scrape: 744


,url,title,content
0,https://nationalgeographic.grid.id/read/134009967/parasut-meraih-sejarah-dunia-penerbangan-sebag...,"Parasut, Meraih Sejarah Dunia Penerbangan Sebagai Sarana Penyelamat",Nationalgeographic.co.id— Gravitasi bumi bisa saja menjadi hal yang fatal bagi makhluk hidup yan...
1,https://nationalgeographic.grid.id/read/134016074/cleopatra-ambisi-kekuasaan-inses-mematikan-di-...,"Cleopatra: Ambisi Kekuasaan, Inses Mematikan di Sejarah Mesir Kuno","Nationalgeographic.co.id— Dalam sejarah Mesir kuno, Cleopatra VII adalah firaun terakhir dari Pt..."
2,https://nationalgeographic.grid.id/read/134017170/hieroglif-bagi-sejarah-mesir-kuno-tulisan-suci...,Hieroglif Bagi Sejarah Mesir Kuno: Tulisan Suci Penemuan Para Dewa,Nationalgeographic.co.id – Hieroglif di sejarah Mesir kuno biasanya diukir pada batu dan digunak...
3,https://nationalgeographic.grid.id/read/134017188/warisan-menyakitkan-tradisi-mengikat-kaki-di-s...,Warisan Menyakitkan Tradisi Mengikat Kaki di Sejarah Tiongkok Kuno,Nationalgeographic.co.id— Pengikatan kaki adalah tradisi di sejarah Tiongkok kuno melibatkan per...
4,https://nationalgeographic.grid.id/read/134017318/sejarah-dunia-dari-mimpi-seorang-nabi-pemberon...,"Sejarah Dunia: dari Mimpi Seorang Nabi, Pemberontakan Taiping Terjadi",Nationalgeographic.co.id— Sejarah dunia pernah mencatat satu bentuk pemberontakan terbesar sepan...


In [ ]:
import pandas as pd
import csv
import re
import os

def clean_csv_rows(input_file, output_file):
    if not os.path.exists(input_file):
        print(f"File {input_file} tidak ditemukan!")
        return

    df = pd.read_csv(input_file)

    prefix_re = re.compile(r'^nationalgeographic\.co\.id\s*(\u2014|\u2013|--|-|\s+)\s*', re.IGNORECASE)

    def clean_text(text, col_name):
        if isinstance(text, str):
            text = re.sub(r'\s+', ' ', text)

            if col_name == 'content':
                text = prefix_re.sub('', text.strip())

            return text.strip()
        return text

    for col in df.columns:
        df[col] = df[col].apply(lambda x: clean_text(x, col))

    df.to_csv(output_file, index=False, encoding='utf-8-sig', quoting=csv.QUOTE_ALL)
    print(f"Berhasil! File yang sudah bersih tanpa prefix disimpan di: {output_file}")
    return df

df_cleaned = clean_csv_rows(OUTPUT_CSV, "natgeo_sejarah_fixed.csv")

if df_cleaned is not None:
    display(df_cleaned.head())

Berhasil! File yang sudah bersih tanpa prefix disimpan di: natgeo_sejarah_fixed.csv


,url,title,content
0,https://nationalgeographic.grid.id/read/134009967/parasut-meraih-sejarah-dunia-penerbangan-sebag...,"Parasut, Meraih Sejarah Dunia Penerbangan Sebagai Sarana Penyelamat",Gravitasi bumi bisa saja menjadi hal yang fatal bagi makhluk hidup yang terjatuh dari ketinggian...
1,https://nationalgeographic.grid.id/read/134016074/cleopatra-ambisi-kekuasaan-inses-mematikan-di-...,"Cleopatra: Ambisi Kekuasaan, Inses Mematikan di Sejarah Mesir Kuno","Dalam sejarah Mesir kuno, Cleopatra VII adalah firaun terakhir dari Ptolemies, sebuah dinasti Yu..."
2,https://nationalgeographic.grid.id/read/134017170/hieroglif-bagi-sejarah-mesir-kuno-tulisan-suci...,Hieroglif Bagi Sejarah Mesir Kuno: Tulisan Suci Penemuan Para Dewa,"Hieroglif di sejarah Mesir kuno biasanya diukir pada batu dan digunakan di kuil, makam, dan monu..."
3,https://nationalgeographic.grid.id/read/134017188/warisan-menyakitkan-tradisi-mengikat-kaki-di-s...,Warisan Menyakitkan Tradisi Mengikat Kaki di Sejarah Tiongkok Kuno,Pengikatan kaki adalah tradisi di sejarah Tiongkok kuno melibatkan perubahan bentuk kaki wanita ...
4,https://nationalgeographic.grid.id/read/134017318/sejarah-dunia-dari-mimpi-seorang-nabi-pemberon...,"Sejarah Dunia: dari Mimpi Seorang Nabi, Pemberontakan Taiping Terjadi",Sejarah dunia pernah mencatat satu bentuk pemberontakan terbesar sepanjang sejarah. Pemberontaka...
